# Antiparallelogram Linkage — Kinematic Visualization
### Elbow mechanism of the tendon-driven tensegrity manipulator

---

## Background and References

This notebook visualizes the kinematics of the **antiparallelogram four-bar linkage**
used as the **elbow joint** in the FAPS tendon-driven manipulator, as described in:

> **Klein, M.** (2023). *Arbeitsraumanalyse, Simulation und Bewegungsplanung eines
> seilgetriebenen robotischen Manipulators.* Master's thesis, Friedrich-Alexander-
> Universität Erlangen-Nürnberg (FAPS), **Section 3.2.1, Figure 3.5**.

Additional references:

- **Uicker, J. J., Pennock, G. R., & Shigley, J. E.** (2011). *Theory of Machines
  and Mechanisms.* 4th ed., Oxford University Press.
  — Four-bar linkage theory, Kennedy's Theorem (§3.4).
- **McCarthy, J. M., & Soh, G. S.** (2010). *Geometric Design of Linkages.* 2nd ed.,
  Springer. — Closure condition derivation (§1.3).
- **Dijksman, E. A.** (1977). On the Cognates of the Antiparallelogram. *ASME J. Eng.
  for Industry*, 99(3). — Rolling centrode ellipses.

---

## Principle of the Antiparallelogram

An **antiparallelogram** (crossed parallelogram) is a four-bar linkage where:

- Frame and coupler have equal length: $|AB| = |DC| = k_e$
- Both connection rods have equal length: $|AD| = |BC| = l_e$
- The connection rods **cross** each other

The connection rods $AD$ and $BC$ are **purely passive** — they carry no
actuators. Instead, antagonistic **tendons** attached at the moving joints
$D$ and $C$ pull the platform symmetrically in both directions.

```
   A ──────── B        ← static frame / base (top)
    \        /
     X──────X          ← crossing point of rods
    /        \
   D ──────── C        ← moving platform / coupler (bottom)
```

### Rolling Centrode Ellipses

A key kinematic property (Dijksman 1977) is that the **centrodes**
(loci of the instantaneous center of rotation) are **ellipses**:

- **Fixed centrode**: foci at $A$, $B$; center at midpoint of $AB$
- **Moving centrode**: congruent ellipse with foci at $D$, $C$; center at coupler midpoint $M$

Both satisfy $|PA| + |PB| = |PD| + |PC| = l_e$, where $P$ is the ICR.
The semi-axes are $a = l_e/2$, $b = \sqrt{l_e^2 - k_e^2}/2$.
The two **congruent** ellipses **roll on each other** — the contact
point is always the current ICR.

---

## Kinematic Derivation

Frame at $y = 0$, platform at $y < 0$. Fixed pivots:

$$A = \left(-\tfrac{k_e}{2},\, 0\right),\quad B = \left(+\tfrac{k_e}{2},\, 0\right)$$

Angle $\theta$ of rod $AD$ (from vertical, clockwise positive):

$$D = A + l_e \begin{pmatrix} \sin\theta \\ -\cos\theta \end{pmatrix}$$

Angle $\varphi$ of rod $BC$ from the closure condition $|DC| = k_e$
(antiparallelogram branch; McCarthy & Soh 2010, §1.3):

$$\varphi = \theta + 2\,\arctan\!\left(\frac{-k_e \cos\theta}{l_e - k_e \sin\theta}\right)$$

**Symmetric equilibrium** (platform horizontal, centered):
$\theta_0 = \arcsin(k_e / l_e)$.

### Elbow angle vs. rod deviation

The **elbow deflection angle** is the rotation of coupler $DC$ from its
horizontal equilibrium position. Due to the nonlinear closure condition,
this is **not** the same as the rod-angle deviation $\delta = \theta - \theta_0$.
The antiparallelogram amplifies and distorts the angular motion.

The animation sweeps the elbow angle linearly through $\pm 75°$
(Klein 2023, §3.2). For each target elbow angle, the corresponding
rod deviation $\delta$ is found by numerically inverting the closure
condition, ensuring perfectly **symmetric** flexion/extension.

---

## Quick Start

> **Kernel:** requires `numpy`, `scipy`, `plotly`.
> Use the `env_isaaclab` conda environment:
> ```bash
> conda run -n env_isaaclab jupyter notebook
> ```
> Or set the VS Code kernel to `env_isaaclab (Python)`.

1. Run **Cell 2 (Parameters)** — adjust `l_e`, `k_e`, `elbow_max_deg` as needed.
2. Run all remaining cells in order.
3. Click **▶ Play** in the animation output.

In [1]:

# =================================================================
# PARAMETERS — adjust here
# =================================================================
import sys
sys.path.insert(0, ".")

import numpy as np
from helpers.linkage import delta_for_elbow, centrode_params

l_e = 150.0           # Rod length [mm] — Klein (2023), §3.2.1
k_e =  60.0           # Joint spacing [mm] — Klein (2023), §3.2.1
elbow_max_deg = 75.0  # Max elbow deflection [deg] — Klein (2023), §3.2
lever_arm_mm = 72.5   # Tendon lever arm [mm]
n_frames = 120        # Animation frames (even number recommended)

# ═══ OPTIONS ═══
show_caption = True    # Set False for thesis-ready figures without titles

# =================================================================
# Derived constants — do not modify
# =================================================================
assert k_e < l_e, f"Mechanism requires l_e > k_e, but l_e={l_e}, k_e={k_e}"
theta_0 = np.arcsin(k_e / l_e)
a_ell, b_ell, c_ell = centrode_params(l_e, k_e)

delta_pos = delta_for_elbow(elbow_max_deg, theta_0, k_e, l_e)
delta_neg = delta_for_elbow(-elbow_max_deg, theta_0, k_e, l_e)
delta_max = max(abs(delta_pos), abs(delta_neg))

print(f"l_e            = {l_e:.1f} mm")
print(f"k_e            = {k_e:.1f} mm")
print(f"k_e / l_e      = {k_e/l_e:.4f}")
print(f"theta_0        = {np.degrees(theta_0):.2f}° (symmetric equilibrium)")
print(f"elbow_max      = ±{elbow_max_deg:.1f}° (coupler rotation limit)")
print(f"δ for +{elbow_max_deg:.0f}° elbow = +{np.degrees(delta_pos):.2f}°")
print(f"δ for -{elbow_max_deg:.0f}° elbow = {np.degrees(delta_neg):.2f}°")
print(f"θ range        = [{np.degrees(theta_0 + delta_neg):.1f}°, "
      f"{np.degrees(theta_0 + delta_pos):.1f}°]")
print(f"\nCentrode ellipse (Dijksman 1977):")
print(f"  a = l_e/2 = {a_ell:.2f} mm")
print(f"  b = √(l_e²−k_e²)/2 = {b_ell:.2f} mm")
print(f"  c = k_e/2 = {c_ell:.2f} mm")
print(f"  ICR at equilibrium: (0, {-b_ell:.2f}) mm")


l_e            = 150.0 mm
k_e            = 60.0 mm
k_e / l_e      = 0.4000
theta_0        = 23.58° (symmetric equilibrium)
elbow_max      = ±75.0° (coupler rotation limit)
δ for +75° elbow = +32.42°
δ for -75° elbow = -42.58°
θ range        = [-19.0°, 56.0°]

Centrode ellipse (Dijksman 1977):
  a = l_e/2 = 75.00 mm
  b = √(l_e²−k_e²)/2 = 68.74 mm
  c = k_e/2 = 30.00 mm
  ICR at equilibrium: (0, -68.74) mm


In [2]:

# =================================================================
# Kinematics Verification
# =================================================================
from helpers.linkage import (
    linkage_positions, instantaneous_center, elbow_angle,
    centrode_ellipse_pts, moving_centrode_pts,
)

# Neutral position & ICR
A0, B0, D0, C0, phi0, M0, th0 = linkage_positions(0.0, k_e, l_e, theta_0)
icr_eq = instantaneous_center(0.0, k_e, l_e, theta_0)

# Fixed centrode ellipse & moving centrode at neutral
fixed_cx, fixed_cy = centrode_ellipse_pts(a_ell, b_ell)
mov_cx_0, mov_cy_0 = moving_centrode_pts(D0, C0, M0, a_ell, b_ell)

# ICR path across the full operating range
delta_range = np.linspace(delta_neg, delta_pos, 600)
icr_list = [instantaneous_center(d, k_e, l_e, theta_0) for d in delta_range]
icr_valid = np.array([p for p in icr_list if p is not None])

# Closure condition verification
max_err = max(abs(np.linalg.norm(
    linkage_positions(d, k_e, l_e, theta_0)[2] -
    linkage_positions(d, k_e, l_e, theta_0)[3]) - k_e)
    for d in delta_range)

# Centrode property: |PA| + |PB| = l_e
sums_AB = np.array([
    np.linalg.norm(p - np.array([-k_e/2, 0])) +
    np.linalg.norm(p - np.array([k_e/2, 0]))
    for p in icr_valid])

ea_at_pos = np.degrees(elbow_angle(delta_pos, k_e, l_e, theta_0))
ea_at_neg = np.degrees(elbow_angle(delta_neg, k_e, l_e, theta_0))

print("=== Verification ===")
print(f"Closure |DC|-k_e max error: {max_err:.2e} mm")
print(f"Centrode |PA|+|PB| = {sums_AB.mean():.6f} +/- {sums_AB.std():.2e}  (theory: {l_e})")
print(f"\nElbow deflection symmetry:")
print(f"  At δ_pos: elbow = {ea_at_pos:+.2f}°")
print(f"  At δ_neg: elbow = {ea_at_neg:+.2f}°")
print(f"\nNeutral position (δ=0):")
print(f"  D = {D0.round(2)},  C = {C0.round(2)},  M = {M0.round(2)}")
print(f"  ICR = ({icr_eq[0]:.4f}, {icr_eq[1]:.4f})")
print(f"  (= bottom vertex of fixed centrode at (0, -{b_ell:.4f}))")


=== Verification ===
Closure |DC|-k_e max error: 4.26e-14 mm
Centrode |PA|+|PB| = 150.000000 +/- 1.87e-14  (theory: 150.0)

Elbow deflection symmetry:
  At δ_pos: elbow = +75.00°
  At δ_neg: elbow = -75.00°

Neutral position (δ=0):
  D = [  30.   -137.48],  C = [ -30.   -137.48],  M = [   0.   -137.48]
  ICR = (0.0000, -68.7386)
  (= bottom vertex of fixed centrode at (0, -68.7386))


In [3]:

# =================================================================
# Static Overview — Antiparallelogram
# =================================================================
from helpers.elbow_plots import antiparallelogram_static_figure

fig = antiparallelogram_static_figure(l_e, k_e, elbow_max_deg, show_caption=show_caption)
fig.show()


In [4]:

# =================================================================
# Animation — Antiparallelogram (0 → +75 → −75 → 0)
# =================================================================
from helpers.elbow_plots import antiparallelogram_animated_figure

fig = antiparallelogram_animated_figure(
    l_e, k_e, elbow_max_deg,
    n_frames=n_frames, show_caption=show_caption,
)
fig.show()


---

## Summary — Antiparallelogram Linkage

| Property | Value |
|---|---|
| Rod length $l_e$ | **150 mm** |
| Joint spacing $k_e$ | **60 mm** |
| Symmetric equilibrium $\theta_0$ | $\arcsin(60/150) \approx 23.6°$ |
| Elbow deflection range | $\pm 75°$ (Klein 2023, §3.2) |
| Rod deviation $\delta$ for $+75°$ | computed numerically (~32°) |
| Rod deviation $\delta$ for $-75°$ | computed numerically (~43°) |
| Centrode semi-axes | $a = 75$ mm, $b \approx 68.7$ mm |
| ICR at equilibrium | $(0,\; -68.7)$ mm |
| Tendon lever arm | $\pm 72.5$ mm (Klein 2023) |

### Rolling Centrode Ellipses

The instantaneous center of rotation (ICR) traces an arc on the
**fixed centrode** — an ellipse with foci at the frame pivots $A$, $B$.
The **moving centrode** is a congruent ellipse with foci at the coupler
endpoints $D$, $C$. Both satisfy $|PA|+|PB| = |PD|+|PC| = l_e$
(Dijksman 1977). The two ellipses roll on each other without slipping.

### Nonlinear Transmission

The antiparallelogram amplifies angular motion: a small rod deviation
$\delta$ produces a larger coupler (elbow) rotation. The transmission
ratio is not constant, so the rod deviations for $+75°$ and $-75°$
elbow angle differ in magnitude. The plots parametrize by **elbow angle**
(symmetric $\pm 75°$) rather than rod deviation $\delta$, ensuring the
depicted poses show equal deflection in both directions.

### Tendon Force Directions

The connection rods $AD$ and $BC$ are purely passive. The elbow is
actuated by two antagonistic tendons (Klein 2023, §3.2.1):
- **T₀**: attached at $D$, pulls toward $B$ (crossing cable) — positive torque
- **T₁**: attached at $C$, pulls toward $A$ (crossing cable) — negative torque

In the IsaacLab tendon model, these map to:
$\tau_{\mathrm{elbow}} = 0.0725 \cdot T_0 - 0.0725 \cdot T_1$


---

## Disc Approximation — Simplified Elbow Model

In simulation (Isaac Lab), the antiparallelogram linkage is replaced by a
**single revolute joint** with a **disc** of radius $r = 72.5\,\text{mm}$
(= tendon lever arm). This is the "elbow approximation" used in the USD
articulation model (`forearm_link` includes `elbow_approx` visual mesh).

### Why the Disc Works

The antiparallelogram's tendon lever arm varies slightly with elbow angle
due to the nonlinear transmission (see centrode ellipse above). The disc
approximation **assumes a constant lever arm** equal to the tendon
attachment radius:

$$\tau_{\mathrm{elbow}} = r \cdot T_0 - r \cdot T_1, \quad r = 72.5\,\text{mm}$$

This yields a **constant** Jacobian transpose $J^T$, simplifying the
tendon-to-torque mapping and enabling standard `ImplicitActuatorCfg`
drives in Isaac Lab.

### Key Differences from Antiparallelogram

| Property | Antiparallelogram | Disc Approximation |
|---|---|---|
| ICR location | Moves along centrode ellipse | **Fixed** at disc center |
| Lever arm | Varies nonlinearly with $\theta$ | **Constant** $r = 72.5$ mm |
| $J^T$ | Configuration-dependent | **Constant** matrix |
| Links | 2 crossed rods (AD, BC) | 1 structural link per side |
| Simulation complexity | Requires 4-bar constraint | Single revolute joint |

The following two plots visualize this simplified model using the same
style and parameters as the antiparallelogram plots above.

In [5]:

# =================================================================
# Static Overview — Disc Approximation
# =================================================================
from helpers.elbow_plots import disc_static_figure

fig = disc_static_figure(l_e, k_e, elbow_max_deg, show_caption=show_caption)
fig.show()


In [6]:

# =================================================================
# Animation — Disc Approximation (0 → +75 → −75 → 0)
# =================================================================
from helpers.elbow_plots import disc_animated_figure

fig = disc_animated_figure(
    l_e, k_e, elbow_max_deg,
    n_frames=n_frames, show_caption=show_caption,
)
fig.show()


In [7]:

# =================================================================
# Animation — Antiparallelogram vs. Disc Approximation (overlaid)
# =================================================================
from helpers.elbow_plots import comparison_animated_figure

fig = comparison_animated_figure(
    l_e, k_e, elbow_max_deg,
    n_frames=n_frames, show_caption=show_caption,
)
fig.show()


In [8]:

# =================================================================
# Comparison — Antiparallelogram vs. Disc Approximation
# Platform centre trajectory and positional deviation
# =================================================================
from helpers.elbow_plots import comparison_static_figure

fig = comparison_static_figure(l_e, k_e, elbow_max_deg, show_caption=show_caption)
fig.show()


---

## Summary — Disc Approximation

| Property | Antiparallelogram | Disc Approximation |
|---|---|---|
| ICR location | Moves along fixed centrode ellipse | **Fixed** at equilibrium ICR $(0,\, -b)$ |
| Lever arm | Varies nonlinearly with $\theta$ | **Constant** $r = k_e / 2$ |
| Jacobian $J^T$ | Configuration-dependent | **Constant** matrix |
| Joint structure | Crossed rods $AD$, $BC$ | Single revolute joint at disc centre |
| Tendon routing | Crossing: T₀ at $D$ toward $B$, T₁ at $C$ toward $A$ | Parallel: T₀ at left tangent, T₁ at right tangent — both straight up |
| Torque model | $\tau = l_e \sin(\alpha) \cdot \Delta T$ (nonlinear $\alpha$) | $\tau = r \cdot \Delta T$ (constant) |
| Simulation | 4-bar closure constraint | `ImplicitActuatorCfg` revolute joint |

### Positional Deviation

Because the disc approximation fixes the ICR, its platform centre follows a
circular arc rather than the true centrode-rolling trajectory. The comparison
plots (cells above) show that this introduces a **positional error** that grows
with elbow angle. At $\pm75°$ the centre-point deviation reaches several
millimetres but the actuator force mapping remains conservative.


---

## References

1. **Klein, M.** (2023). *Arbeitsraumanalyse, Simulation und Bewegungsplanung
   eines seilgetriebenen robotischen Manipulators.* Master's thesis, FAU
   Erlangen-Nürnberg (FAPS). §3.2, **§3.2.1**, Fig. 3.5.

2. **Uicker, J. J., Pennock, G. R., & Shigley, J. E.** (2011). *Theory of
   Machines and Mechanisms.* 4th ed., Oxford University Press.
   (Ch. 2: four-bar linkage; §3.4: ICR, Kennedy's Theorem)

3. **McCarthy, J. M., & Soh, G. S.** (2010). *Geometric Design of Linkages.*
   2nd ed., Springer. (§1.3: closure condition)

4. **Dijksman, E. A.** (1977). On the Cognates of the Antiparallelogram.
   *ASME J. Eng. for Industry*, 99(3). (Rolling centrode ellipses)
